In [ ]:
!pip install -U evaluate sacrebleu rouge-score

In [ ]:
from datasets import load_dataset, Dataset
import evaluate
import pandas as pd
from tqdm import tqdm
import os
os.environ["HF_TOKEN"] = "hf_token" # removed for safety reasons

from huggingface_hub import login
login(os.environ["HF_TOKEN"])

# PRED_DATASET = "businessrules/Qwen_base_tuned_exp16_results"
# Results_dataset = "businessrules/Qwen_base_eval_bleu_rouge_exp16"

In [ ]:
from datasets import load_dataset, Dataset
import evaluate
import pandas as pd

# =========================================================
# 1. Load evaluation dataset
# =========================================================
eval_dataset = load_dataset(PRED_DATASET, split = "train")

# =========================================================
# 2. Load metrics
# =========================================================
bleu = evaluate.load("sacrebleu")
rouge = evaluate.load("rouge")

# =========================================================
# 3. Prepare references and predictions
# =========================================================
references = [[x] for x in eval_dataset["golden_business_rule"]]

# base_predictions = eval_dataset["base_model_prediction"]
ft_predictions   = eval_dataset["finetuned_model_prediction"]

# =========================================================
# 4. BLEU
# =========================================================
# base_bleu = bleu.compute(
#     predictions=base_predictions,
#     references=references
# )

ft_bleu = bleu.compute(
    predictions=ft_predictions,
    references=references
)

# =========================================================
# 5. ROUGE
# =========================================================
# base_rouge = rouge.compute(
#     predictions=base_predictions,
#     references=eval_dataset["golden_business_rule"]
# )

ft_rouge = rouge.compute(
    predictions=ft_predictions,
    references=eval_dataset["golden_business_rule"]
)

# =========================================================
# 6. Aggregate results table
# =========================================================
summary_results = [
    # {
    #     "model": "base",
    #     "BLEU": base_bleu["score"] / 100,
    #     "ROUGE-1": base_rouge["rouge1"],
    #     "ROUGE-2": base_rouge["rouge2"],
    #     "ROUGE-L": base_rouge["rougeL"],
    # },
    {
        "model": "finetuned",
        "BLEU": ft_bleu["score"] / 100,
        "ROUGE-1": ft_rouge["rouge1"],
        "ROUGE-2": ft_rouge["rouge2"],
        "ROUGE-L": ft_rouge["rougeL"],
    }
]

results_df = pd.DataFrame(summary_results)
print(results_df)

metrics_dataset = Dataset.from_pandas(results_df)

metrics_dataset.push_to_hub(Results_dataset)


In [ ]:
# BLEU and ROUGE for GPT-4.1
from datasets import load_dataset, Dataset
import evaluate
import pandas as pd

# =========================================================
# 1. Load evaluation dataset
# =========================================================
PRED_DATASET = "businessrules/GPT4_baseline_results"
Results_dataset = "businessrules/GPT4_eval_bleu_rouge"

eval_dataset = load_dataset(PRED_DATASET, split="train")

# =========================================================
# 2. Load metrics
# =========================================================
bleu = evaluate.load("sacrebleu")
rouge = evaluate.load("rouge")

# =========================================================
# 3. Prepare references and predictions
# =========================================================
references = [[x] for x in eval_dataset["golden_business_rule"]]
gpt4_predictions = eval_dataset["gpt4_prediction"]

# =========================================================
# 4. BLEU
# =========================================================
gpt4_bleu = bleu.compute(
    predictions=gpt4_predictions,
    references=references
)

# =========================================================
# 5. ROUGE
# =========================================================
gpt4_rouge = rouge.compute(
    predictions=gpt4_predictions,
    references=eval_dataset["golden_business_rule"]
)

# =========================================================
# 6. Aggregate results table
# =========================================================
summary_results = [
    {
        "model": "gpt4",
        "BLEU": gpt4_bleu["score"] / 100,
        "ROUGE-1": gpt4_rouge["rouge1"],
        "ROUGE-2": gpt4_rouge["rouge2"],
        "ROUGE-L": gpt4_rouge["rougeL"],
    }
]

results_df = pd.DataFrame(summary_results)
print(results_df)

metrics_dataset = Dataset.from_pandas(results_df)
metrics_dataset.push_to_hub(Results_dataset)